# 03 - QLoRA fine-tuning (Colab T4)

> Runtime -> Change runtime type -> T4 GPU. Then install the GPU stack:
> `pip install unsloth trl peft transformers datasets accelerate bitsandbytes`

QLoRA = a 4-bit (NF4) base + a small LoRA adapter trained on top. An 8B model
fine-tunes in ~8-12GB, which fits a free T4. We train rank 16 / alpha 32 on all
linear layers, the 2026 default.

In [ ]:
import sys; sys.path.insert(0, '../src')
from qlora_lab import schema, synth, dataset, predict, evaluate, train, serve, agent

### Configure and train
Every knob is commented in `src/qlora_lab/train.py`.

In [ ]:
cfg = train.TrainConfig(
    base_model='unsloth/Qwen3-8B-bnb-4bit',
    lora_r=16, lora_alpha=32, epochs=3, learning_rate=2e-4,
    output_dir='outputs/adapter',
)
adapter_dir = train.train('../data/train.jsonl', cfg)
print('adapter saved to', adapter_dir)  # ~tens of MB, not the whole model

### Benchmark multiple bases
The resume line says *benchmarking Qwen3, LLaMA3, Gemma, GPT-OSS*. That is
literally running this cell with different `base_model` values on the same data
and picking the Pareto-best on quality and cost - not defaulting to the biggest.

In [ ]:
# for base in ['unsloth/Qwen3-8B-bnb-4bit', 'unsloth/llama-3.1-8b-bnb-4bit',
#              'unsloth/gemma-2-9b-bnb-4bit']:
#     train.train('../data/train.jsonl', train.TrainConfig(base_model=base,
#                 output_dir=f'outputs/{base.split("/")[-1]}'))